# Tool 3 — QA Scorer: manual test notebook

Exercises the 9-parameter **weighted** rubric scorer in `src/tools/qa_scorer.py`.

**Import caveat:** importing `qa_scorer` transitively imports `policy_search`,
which builds *and probes* its LLM client at import time. So for the
`local_lmstudio` provider, **every cell below needs the LM Studio server
reachable just to import the module** — but only the final live cell (h) makes
actual scoring LLM calls. Cells (a)–(g) exercise the deterministic machinery
(weights, category, `sla_met`, weighted arithmetic, guards) and make **no**
scoring call.

### (a) Imports + config — provider, structured-output model, rubric

In [ ]:
import os, sys

# Make `import src...` work when the kernel's cwd is notebooks/.
sys.path.insert(0, os.path.abspath(".."))

from src.config import CONFIG
import src.tools.qa_scorer as q

prov = CONFIG["active_provider"]
pconf = CONFIG["providers"][prov]
print(f"Active provider          : {prov}")
print(f"Main model               : {pconf['model']}")
print(f"Structured-output model  : {pconf.get('structured_output_model', '(main model)')}")
print(f"Structured-output method : {pconf.get('structured_output_method', '(default)')}")
print(f"PASS threshold           : {int(q.PASS_FRACTION * 100)}% of the weighted total")
print(f"Gating parameters        : {list(q.GATING_PARAMS)}  (a FAIL here forces overall FAIL)")
print(f"Rubric parameters ({len(q.PARAM_NAMES)}) : {q.PARAM_NAMES}")

### (b) Schema shape — 8 LLM-judged params, `sla_met` deterministic & excluded

The LLM only judges the 8 `LLM_RUBRIC` parameters (a `QAResult`). `sla_met` is
read from the call record, so it must **not** be a field on the structured-output
schema, yet it must be the 9th entry in `PARAM_NAMES`.

In [ ]:
llm_params = [name for name, _ in q.LLM_RUBRIC]
schema_fields = list(q.QAResult.model_fields.keys())

print("LLM_RUBRIC params (8):", llm_params)
print("QAResult schema fields:", schema_fields)
print("PARAM_NAMES (9):", q.PARAM_NAMES)

assert schema_fields == llm_params, "QAResult must contain exactly the 8 LLM params"
assert "sla_met" not in schema_fields, "sla_met must NOT be in the LLM schema"
assert q.PARAM_NAMES == llm_params + ["sla_met"], "sla_met must be the 9th rubric param"
print("\nOK: sla_met is deterministic (not judged) and is the 9th rubric parameter.")

### (c) Weights — every category column sums to 100

`data/qa_weights.csv` is the source of truth; `_weights()` loads it into
`{category: {parameter: weight}}`. Each category must sum to 100 so the weighted
score reads directly as a percentage.

In [ ]:
import pandas as pd

weights = q._weights()
print("Weight table (data/qa_weights.csv):")
print(pd.read_csv(q.WEIGHTS_CSV).to_string(index=False))

print()
for cat in ("inbound", "outbound", "email"):
    total = sum(weights[cat][name] for name in q.PARAM_NAMES)
    print(f"  {cat:<8} sum = {total}  (professional_tone={weights[cat]['professional_tone']}, "
          f"call_classification={weights[cat]['call_classification']})")
    assert total == 100, f"{cat} column must sum to 100"

assert all(weights[c]["professional_tone"] == 15 for c in ("inbound", "outbound", "email")), \
    "professional_tone should be weighted 15"
print("\nOK: all three categories sum to 100; professional_tone raised to 15.")

### (d) Category selection — email overrides direction

The weighting category is `email` for email-channel rows regardless of direction,
otherwise the row's `direction`. An unknown/blank value falls back to `inbound`.

In [ ]:
combos = [
    ("inbound", "phone"), ("outbound", "phone"),
    ("inbound", "chat"), ("outbound", "chat"),
    ("inbound", "email"), ("outbound", "email"),
    ("", ""),  # unknown -> default inbound
]
for direction, channel in combos:
    print(f"  direction={direction or '(blank)':<8} channel={channel or '(blank)':<6} "
          f"-> category={q._category(direction, channel)}")

assert q._category("outbound", "email") == "email", "email must override direction"
assert q._category("", "") == "inbound", "blank must default to inbound"
print("\nOK: email overrides direction; blanks default to inbound.")

### (e) Deterministic `sla_met` + category from the call record

`_sla_result_and_category(call_id)` reads `data/calls.csv`: `sla_met == "Yes"` ->
PASS. A missing/unknown/blank `call_id` FAILs with a "no SLA data available" note
(same defensive pattern as policy_compliance). No LLM involved.

In [ ]:
# CALL-1001 good (Yes, phone/inbound); CALL-1006 bad (No, phone/outbound);
# CALL-1008 email (device/inbound/email); CALL-9999 unknown; "" no call_id.
for cid in ("CALL-1001", "CALL-1006", "CALL-1008", "CALL-9999", ""):
    pr, cat = q._sla_result_and_category(cid)
    label = cid or "(empty)"
    print(f"  {label:<10} sla_met={'PASS' if pr.passed else 'FAIL':<4} "
          f"category={cat:<8} note={pr.justification!r}")

pr, _ = q._sla_result_and_category("CALL-1001"); assert pr.passed
pr, _ = q._sla_result_and_category("CALL-1006"); assert not pr.passed
_, cat = q._sla_result_and_category("CALL-1008"); assert cat == "email"
pr, _ = q._sla_result_and_category("CALL-9999"); assert not pr.passed and "no SLA data" in pr.justification
print("\nOK: sla_met resolves Yes/No from the record and FAILs-with-note when unknown.")

### (f) Weighted-score arithmetic + the professional_tone gate (no scoring LLM call)

`_format_report` sums the PASSED parameters' weights. We feed it fabricated
`ParamResult`s so the arithmetic is checked without invoking the model.
`professional_tone` is a **gating** parameter (`GATING_PARAMS`): a FAIL there caps
the overall verdict at FAIL *regardless* of the weighted score.

In [ ]:
from src.tools.qa_scorer import ParamResult

def all_pass():
    return {name: ParamResult(passed=True, justification="ok") for name in q.PARAM_NAMES}

w = q._weights()["inbound"]

# Scenario 1 — a flawless inbound call: every parameter passes -> 100/100 PASS.
r1 = all_pass()
print(q._format_report(r1, "CALL-DEMO-A", "inbound"))

# Scenario 2 — GATE: fail ONLY professional_tone(15). Score is 85/100 (well above
# the 60% bar), but the tone gate forces the overall verdict to FAIL.
r2 = all_pass()
r2["professional_tone"] = ParamResult(passed=False, justification="rude/dismissive")
report2 = q._format_report(r2, "CALL-DEMO-B", "inbound")
print("\n" + report2)
earned2 = sum(w[n] for n in q.PARAM_NAMES if r2[n].passed)
assert earned2 == 85, f"expected 85, got {earned2}"
assert "— FAIL" in report2 and "automatic FAIL" in report2, "tone FAIL must gate to overall FAIL"

# Scenario 3 — NON-gating failure: fail only proper_closure(5) -> 95/100, tone
# passes, so it still PASSes (the gate only trips on gating params).
r3 = all_pass()
r3["proper_closure"] = ParamResult(passed=False, justification="abrupt close")
report3 = q._format_report(r3, "CALL-DEMO-C", "inbound")
print("\n" + report3)
assert "— PASS" in report3, "a non-gating failure above threshold should still PASS"

print(f"\nOK: tone FAIL gates 85/100 -> FAIL; a non-gating 95/100 -> PASS.")

### (g) Guards — empty transcript + policy-context force-fail (no scoring LLM call)

`score_call` returns a message (never raises) on an empty transcript, and when no
policy context is available it both marks the prompt and force-FAILs
`policy_compliance` in code. Here we verify the empty-transcript guard and the
prompt marker without a model call.

In [ ]:
print("Empty transcript ->", repr(q.score_call("")))
assert q.score_call("").startswith("Cannot score")

prompt_no_ctx = q._build_prompt("some transcript", "", policy_available=False)
assert "NO POLICY CONTEXT AVAILABLE" in prompt_no_ctx
assert q.NO_POLICY_NOTE in prompt_no_ctx
print("\nOK: empty-transcript guard returns a message; the no-policy marker is in the prompt.")
print("(In score_call, policy_compliance is additionally force-FAILed in code when context is absent.)")

### (h) LIVE end-to-end — **SLOW**, needs the provider up

This makes real calls: Tool 1 retrieval (heavy model answer-gen) **plus** the
structured scorer, per call — roughly a couple of minutes each on the local
model. Scores a clean call (CALL-1001) and a known-bad rude call (CALL-1016),
letting the tool self-retrieve `policy_context`.

In [ ]:
import time
import pandas as pd

df = pd.read_csv("../data/calls.csv", dtype=str).set_index("call_id")
for cid in ("CALL-1001", "CALL-1016"):
    row = df.loc[cid]
    print("\n" + "#" * 68)
    print(f"# {cid}  ({row.call_type}/{row.direction}/{row.channel}, sla_met={row.sla_met})")
    print("#" * 68)
    t = time.time()
    report = q.score_call(row.transcript, call_id=cid)
    print(f"[took {time.time() - t:.0f}s]")
    print(report)